In [1]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

spark = configure_spark_with_delta_pip(
    SparkSession.builder
        .appName("Python Spark SQL basic example")
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
        .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
        .config("spark.driver.memory", "4g")
        .config("spark.executor.memory", "4g")
        .config("spark.sql.codegen.wholeStage", "false")
).getOrCreate()


23/10/16 01:42:18 WARN Utils: Your hostname, codespaces-7c531b resolves to a loopback address: 127.0.0.1; using 172.16.5.4 instead (on interface eth0)
23/10/16 01:42:18 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/usr/local/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
io.delta#delta-core_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-228d66b0-bcbc-47f2-85f6-31923fe84bad;1.0
	confs: [default]
	found io.delta#delta-core_2.12;2.4.0 in central
	found io.delta#delta-storage;2.4.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 257ms :: artifacts dl 9ms
	:: modules in use:
	io.delta#delta-core_2.12;2.4.0 from central in [default]
	io.delta#delta-storage;2.4.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   0   |   0   |   0  

In [2]:
from urllib.parse import urlparse
from pyspark import SparkFiles
from pyspark.sql import DataFrame

def load_dataset(url: str) -> DataFrame:
    spark.sparkContext.addFile(url)
    file = urlparse(url).path.split('/')[-1]
    df = spark.read.csv(SparkFiles.get(file), header=True, inferSchema=True)
    return df

# iris data url
url = "https://gist.githubusercontent.com/netj/8836201/raw/6f9306ad21398ea43cba4f7d537619d0e07d5ae3/iris.csv"
df = load_dataset(url)
df.show()


+------------+-----------+------------+-----------+-------+
|sepal.length|sepal.width|petal.length|petal.width|variety|
+------------+-----------+------------+-----------+-------+
|         5.1|        3.5|         1.4|        0.2| Setosa|
|         4.9|        3.0|         1.4|        0.2| Setosa|
|         4.7|        3.2|         1.3|        0.2| Setosa|
|         4.6|        3.1|         1.5|        0.2| Setosa|
|         5.0|        3.6|         1.4|        0.2| Setosa|
|         5.4|        3.9|         1.7|        0.4| Setosa|
|         4.6|        3.4|         1.4|        0.3| Setosa|
|         5.0|        3.4|         1.5|        0.2| Setosa|
|         4.4|        2.9|         1.4|        0.2| Setosa|
|         4.9|        3.1|         1.5|        0.1| Setosa|
|         5.4|        3.7|         1.5|        0.2| Setosa|
|         4.8|        3.4|         1.6|        0.2| Setosa|
|         4.8|        3.0|         1.4|        0.1| Setosa|
|         4.3|        3.0|         1.1| 

In [10]:
df.write.format("parquet").mode("overwrite").save("/tmp/data/parquet")
df.write.format("parquet").mode("overwrite").save("/tmp/data/delta")
df.write.format("csv").mode("overwrite").option("header", True).save("/tmp/data/csv")
df.write.format("json").mode("overwrite").save("/tmp/data/json")
df.write.format("orc").mode("overwrite").save("/tmp/data/orc")


In [12]:
import pandas as pd

pd.read_csv(url).to_csv("/tmp/standalone_iris.csv", index=False)


In [16]:
import os
file = os.path.abspath(os.path.join(os.path.dirname("__file__"), "../output.sql"))
print(f"Reading query file: {file}")


with open(file, "r") as f:
    queries = f.readlines()

for query in queries:
    if not query.startswith("-- ") and not query.startswith("\n"):
        print(query)
        spark.sql(query)


Reading query file: /workspaces/unity-catalog-helper/output.sql
CREATE SCHEMA IF NOT EXISTS `demo_schema`;

CREATE TABLE IF NOT EXISTS `demo_schema`.`csv` USING CSV OPTIONS (path = '/tmp/data/csv', header = 'true');

CREATE TABLE IF NOT EXISTS `demo_schema`.`delta` USING PARQUET LOCATION '/tmp/data/delta';

CREATE TABLE IF NOT EXISTS `demo_schema`.`orc` USING ORC LOCATION '/tmp/data/orc';

CREATE TABLE IF NOT EXISTS `demo_schema`.`parquet` USING PARQUET LOCATION '/tmp/data/parquet';

CREATE TABLE IF NOT EXISTS `demo_schema`.`json` USING JSON LOCATION '/tmp/data/json';

CREATE TABLE IF NOT EXISTS `demo_schema`.`some_table` USING CSV OPTIONS (path = '/tmp/data/csv', header = 'true');

CREATE TABLE IF NOT EXISTS `demo_schema`.`some_table2` USING ORC LOCATION '/tmp/data/orc';

CREATE TABLE IF NOT EXISTS `demo_schema`.`some_table3` USING PARQUET LOCATION '/tmp/data/delta';

CREATE TABLE IF NOT EXISTS `demo_schema`.`some_table4` USING PARQUET LOCATION '/tmp/data/parquet';

CREATE TABLE IF NO

In [17]:
spark.catalog.listDatabases()


[Database(name='default', catalog='spark_catalog', description='default database', locationUri='file:/workspaces/unity-catalog-helper/research/spark-warehouse'),
 Database(name='demo_schema', catalog='spark_catalog', description='', locationUri='file:/workspaces/unity-catalog-helper/research/spark-warehouse/demo_schema.db')]

In [18]:
spark.catalog.listTables("default")


[]

In [19]:
spark.catalog.listTables("demo_schema")


[Table(name='csv', catalog='spark_catalog', namespace=['demo_schema'], description=None, tableType='EXTERNAL', isTemporary=False),
 Table(name='delta', catalog='spark_catalog', namespace=['demo_schema'], description=None, tableType='EXTERNAL', isTemporary=False),
 Table(name='json', catalog='spark_catalog', namespace=['demo_schema'], description=None, tableType='EXTERNAL', isTemporary=False),
 Table(name='orc', catalog='spark_catalog', namespace=['demo_schema'], description=None, tableType='EXTERNAL', isTemporary=False),
 Table(name='parquet', catalog='spark_catalog', namespace=['demo_schema'], description=None, tableType='EXTERNAL', isTemporary=False),
 Table(name='some_table', catalog='spark_catalog', namespace=['demo_schema'], description=None, tableType='EXTERNAL', isTemporary=False),
 Table(name='some_table2', catalog='spark_catalog', namespace=['demo_schema'], description=None, tableType='EXTERNAL', isTemporary=False),
 Table(name='some_table3', catalog='spark_catalog', namespace

In [27]:
for table in spark.catalog.listTables("demo_schema"):
    print(f"Table: demo_schema.{table.name}")
    spark.read.table(f"demo_schema.{table.name}").show(truncate=False, n=2)


Table: demo_schema.csv
+------------+-----------+------------+-----------+-------+
|sepal.length|sepal.width|petal.length|petal.width|variety|
+------------+-----------+------------+-----------+-------+
|5.1         |3.5        |1.4         |0.2        |Setosa |
|4.9         |3.0        |1.4         |0.2        |Setosa |
+------------+-----------+------------+-----------+-------+
only showing top 2 rows

Table: demo_schema.delta
+------------+-----------+------------+-----------+-------+
|sepal.length|sepal.width|petal.length|petal.width|variety|
+------------+-----------+------------+-----------+-------+
|5.1         |3.5        |1.4         |0.2        |Setosa |
|4.9         |3.0        |1.4         |0.2        |Setosa |
+------------+-----------+------------+-----------+-------+
only showing top 2 rows

Table: demo_schema.json
+------------+-----------+------------+-----------+-------+
|petal.length|petal.width|sepal.length|sepal.width|variety|
+------------+-----------+------------